# 06 — Clustering & Dimensionality Reduction

Unsupervised methods: grouping similar samples and reducing feature dimensionality.

## K-Means

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=3, n_init=10, random_state=42)
km.fit(X_train)
km.labels_             # cluster assignment per training sample
km.predict(X_test)     # assign new points to nearest cluster
km.cluster_centers_
km.inertia_             # sum of squared distances to nearest center (used for elbow method)


## Choosing k: Elbow Method & Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score

inertias, sil_scores = [], []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_train)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_train, km.labels_))

plt.plot(range(2, 11), inertias, marker="o")
plt.xlabel("k"); plt.ylabel("inertia"); plt.title("Elbow method")
plt.show()


## DBSCAN (density-based, no need to pick k, finds arbitrary shapes)

In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.5, min_samples=5)
labels = db.fit_predict(X_train)
(labels == -1).sum()   # -1 label marks noise/outlier points


## Hierarchical (Agglomerative) Clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

agg = AgglomerativeClustering(n_clusters=3, linkage="ward")
agg.fit_predict(X_train)

# dendrogram visualization
Z = linkage(X_train, method="ward")
dendrogram(Z, truncate_mode="level", p=5)
plt.show()


## Gaussian Mixture Models (soft/probabilistic clustering)

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=42)
gmm.fit(X_train)
gmm.predict(X_test)             # hard cluster assignment
gmm.predict_proba(X_test)       # soft assignment probabilities per component
gmm.bic(X_train)                # Bayesian Information Criterion, lower is better, for choosing n_components


## PCA (Principal Component Analysis)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train)
pca.explained_variance_ratio_        # fraction of variance captured per component
pca.explained_variance_ratio_.cumsum()

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train, cmap="viridis")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

# choosing n_components to retain a variance threshold
pca_95 = PCA(n_components=0.95)   # keep enough components for 95% variance
X_reduced = pca_95.fit_transform(X_train)


## t-SNE (nonlinear, visualization-only)

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_train)   # no .transform() for new data — must refit each time
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_train, cmap="viridis")
plt.show()


## Truncated SVD (PCA alternative for sparse data, e.g. text/TF-IDF)

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state=42)
X_svd = svd.fit_transform(X_train_sparse)   # works directly on sparse matrices, unlike PCA


## Common Gotchas
- K-Means assumes roughly spherical, similarly-sized clusters — DBSCAN or GMM handle irregular shapes/densities better.
- Always scale features before K-Means/PCA — unscaled features with large ranges dominate the distance/variance calculations.
- `t-SNE` has no `.transform()` method for new data and is stochastic — good for visualization, not for a reusable feature-reduction pipeline.
- `silhouette_score` is expensive on large datasets (O(n^2)); subsample for a quick estimate on big data.